In [1]:
import pandas as pd
import duckdb

In [2]:
trips ='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet'
zones = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv'

df_zone = duckdb.sql(f"""
    SELECT *
    FROM read_csv('{zones}')
    LIMIT 5
""").fetchdf()

df_trips = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{trips}')
    LIMIT 0
""").fetchdf()

display(df_zone)
display(df_trips.T)

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


""
VendorID
lpep_pickup_datetime
lpep_dropoff_datetime
store_and_fwd_flag
RatecodeID
PULocationID
DOLocationID
passenger_count
trip_distance
fare_amount


In [3]:
trips ='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet'
zones = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv'

df = duckdb.sql(f"""
    SELECT
        TRY_CAST(lpep_pickup_datetime AS Date) AS PickUpDate,
        b.Zone AS PickUpZone,
        c.Zone as DropOffZone,
        tip_amount AS Tip
    FROM read_parquet('{trips}') a
    JOIN read_csv('{zones}') b ON a.PULocationID == b.LocationID
    JOIN read_csv('{zones}') c ON a.DOLocationID == c.LocationID
    WHERE
        PickUpZone = 'East Harlem North'
        AND YEAR(PickUpDate) = 2025
        AND MONTH(PickUpDate) = 11
    ORDER BY Tip DESC
    LIMIT 5
""").fetchdf().set_index('PickUpDate')

display(df)

,PickUpZone,DropOffZone,Tip
PickUpDate,,,
2025-11-30,East Harlem North,Yorkville West,81.89
2025-11-23,East Harlem North,LaGuardia Airport,50.00
2025-11-29,East Harlem North,East Harlem North,45.00
2025-11-24,East Harlem North,Long Island City/Queens Plaza,34.25
2025-11-15,East Harlem North,NA,28.90


In [4]:
trips ='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet'
zones = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv'

df = duckdb.sql(f"""
    SELECT
        TRY_CAST(lpep_pickup_datetime AS Date) AS PickUpDate,
        b.Zone AS PickUpZone,
        c.Zone as DropOffZone,
        tip_amount AS Tip,
        RANK() OVER (ORDER BY tip_amount DESC) AS TipRank
    FROM read_parquet('{trips}') a
    JOIN read_csv('{zones}') b ON a.PULocationID == b.LocationID
    JOIN read_csv('{zones}') c ON a.DOLocationID == c.LocationID
    WHERE
        PickUpZone = 'East Harlem North'
        AND YEAR(PickUpDate) = 2025
        AND MONTH(PickUpDate) = 11
    ORDER BY TipRank ASC
    LIMIT 5
""").fetchdf().set_index('PickUpDate')

display(df)

,PickUpZone,DropOffZone,Tip,TipRank
PickUpDate,,,,
2025-11-30,East Harlem North,Yorkville West,81.89,1
2025-11-23,East Harlem North,LaGuardia Airport,50.00,2
2025-11-29,East Harlem North,East Harlem North,45.00,3
2025-11-24,East Harlem North,Long Island City/Queens Plaza,34.25,4
2025-11-15,East Harlem North,NA,28.90,5
